# Hippocampus Data Split Generator and Validator

This notebook creates train/validation/test splits from hippocampus data using only healthy samples (first 510 entries) and includes comprehensive tests to verify data quality.

## 1. Import Required Libraries

In [1]:
import os
import json
import random
import torch
import unittest
from typing import List

## 2. Load and Process Data

In [2]:
# Load labels.pt file
labels_path = '../../../hippocampus_data_tle_ms_age_and_0_1/hippoData_regstrd_disease_reconstrct_ply/labels.pt'
labels = torch.load(labels_path)

# Extract filenames from labels - take only first 510 entries
file_ids = list(labels.keys())[:510]
obj_files = [f"{file_id}.obj" for file_id in file_ids]

print(f"Loaded {len(file_ids)} file IDs from labels.pt")
print(f"Sample file IDs: {file_ids[:5]}")
print(f"Sample obj files: {obj_files[:5]}")

Loaded 510 file IDs from labels.pt
Sample file IDs: ['ab300_001', 'ab300_002', 'ab300_003', 'ab300_004', 'ab300_005']
Sample obj files: ['ab300_001.obj', 'ab300_002.obj', 'ab300_003.obj', 'ab300_004.obj', 'ab300_005.obj']


In [8]:
# Analyze the value range of index 0 across all 510 entries
import numpy as np

print("Analyzing labels data structure and value ranges...")
print("=" * 50)

# First, let's examine the structure of one label entry
sample_label = labels.get(file_ids[0])
print(f"Sample label structure for '{file_ids[0]}':")
print(f"Type: {type(sample_label)}")
print(f"Shape: {sample_label.shape if hasattr(sample_label, 'shape') else 'N/A'}")
print(f"Sample values: {sample_label}")
print()

# Extract all values at index 0 from all 510 entries
index_0_values = []
for file_id in file_ids:
    label_data = labels.get(file_id)
    if label_data is not None:
        # Get the value at index 0
        if isinstance(label_data, (list, tuple, np.ndarray)) and len(label_data) > 0:
            index_0_values.append(label_data[0])
        elif hasattr(label_data, 'item'):  # For torch tensors or single values
            index_0_values.append(label_data.item() if hasattr(label_data, 'item') else label_data)

# Convert to numpy array for easier analysis
if index_0_values:
    index_0_array = np.array(index_0_values)
    
    print(f"Analysis of index 0 values across {len(index_0_values)} entries:")
    print(f"Data type: {index_0_array.dtype}")
    print(f"Shape: {index_0_array.shape}")
    print(f"Min value: {np.min(index_0_array)}")
    print(f"Max value: {np.max(index_0_array)}")
    print(f"Mean value: {np.mean(index_0_array):.4f}")
    print(f"Std deviation: {np.std(index_0_array):.4f}")
    print(f"Unique values: {len(np.unique(index_0_array))}")
    print(f"Value range: [{np.min(index_0_array)}, {np.max(index_0_array)}]")
    print()
    
    # Show distribution of values
    unique_vals, counts = np.unique(index_0_array, return_counts=True)
    print("Value distribution (first 20 most common):")
    sorted_indices = np.argsort(counts)[::-1]  # Sort by count descending
    for i in range(min(20, len(unique_vals))):
        idx = sorted_indices[i]
        print(f"  Value {unique_vals[idx]}: {counts[idx]} occurrences")
        
else:
    print("No valid index 0 values found in the labels data.")

Analyzing labels data structure and value ranges...
Sample label structure for 'ab300_001':
Type: <class 'numpy.ndarray'>
Shape: (2,)
Sample values: [0.34117647 0.        ]

Analysis of index 0 values across 510 entries:
Data type: float64
Shape: (510,)
Min value: 0.0
Max value: 1.0
Mean value: 0.3534
Std deviation: 0.2576
Unique values: 81
Value range: [0.0, 1.0]

Value distribution (first 20 most common):
  Value 0.047058823529411764: 18 occurrences
  Value 0.18823529411764706: 18 occurrences
  Value 0.011764705882352941: 17 occurrences
  Value 0.03529411764705882: 17 occurrences
  Value 0.15294117647058825: 14 occurrences
  Value 0.1176470588235294: 13 occurrences
  Value 0.19999999999999996: 12 occurrences
  Value 0.16470588235294115: 12 occurrences
  Value 0.22352941176470587: 12 occurrences
  Value 0.1764705882352941: 11 occurrences
  Value 0.30588235294117644: 11 occurrences
  Value 0.12941176470588234: 11 occurrences
  Value 0.3764705882352941: 11 occurrences
  Value 0.05882352

## 3. Test Functions

In [9]:
def test_entry_count(file_ids: List[str], expected_count: int = 510) -> tuple:
    """Test that we have exactly the expected number of entries."""
    actual_count = len(file_ids)
    passed = actual_count == expected_count
    message = f"Expected {expected_count} entries, got {actual_count}"
    return passed, message

def test_no_forbidden_words(file_ids: List[str], forbidden_words: List[str] = ['ms', 'tle']) -> tuple:
    """Test that no file ID contains forbidden words."""
    problematic_files = []
    
    for file_id in file_ids:
        file_id_lower = file_id.lower()
        for word in forbidden_words:
            if word.lower() in file_id_lower:
                problematic_files.append((file_id, word))
    
    passed = len(problematic_files) == 0
    if passed:
        message = f"✓ No entries contain forbidden words: {forbidden_words}"
    else:
        message = f"✗ Found {len(problematic_files)} entries with forbidden words: {problematic_files[:10]}"  # Show first 10
    
    return passed, message

def test_obj_files_format(obj_files: List[str]) -> tuple:
    """Test that all obj files have proper .obj extension."""
    invalid_files = [f for f in obj_files if not f.endswith('.obj')]
    passed = len(invalid_files) == 0
    message = f"All {len(obj_files)} files have .obj extension" if passed else f"Found {len(invalid_files)} files without .obj extension"
    return passed, message

## 4. Run All Tests

In [10]:
print("=" * 60)
print("RUNNING DATA VALIDATION TESTS")
print("=" * 60)

# Test 1: Entry count
passed1, message1 = test_entry_count(file_ids)
print(f"Test 1 - Entry Count: {'✓ PASSED' if passed1 else '✗ FAILED'}")
print(f"  {message1}")
print()

# Test 2: No forbidden words
passed2, message2 = test_no_forbidden_words(file_ids)
print(f"Test 2 - No Forbidden Words: {'✓ PASSED' if passed2 else '✗ FAILED'}")
print(f"  {message2}")
print()

# Test 3: OBJ file format
passed3, message3 = test_obj_files_format(obj_files)
print(f"Test 3 - OBJ File Format: {'✓ PASSED' if passed3 else '✗ FAILED'}")
print(f"  {message3}")
print()

# Summary
all_tests_passed = all([passed1, passed2, passed3])
print("=" * 60)
print(f"OVERALL RESULT: {'✓ ALL TESTS PASSED' if all_tests_passed else '✗ SOME TESTS FAILED'}")
print("=" * 60)

RUNNING DATA VALIDATION TESTS
Test 1 - Entry Count: ✓ PASSED
  Expected 510 entries, got 510

Test 2 - No Forbidden Words: ✓ PASSED
  ✓ No entries contain forbidden words: ['ms', 'tle']

Test 3 - OBJ File Format: ✓ PASSED
  All 510 files have .obj extension

OVERALL RESULT: ✓ ALL TESTS PASSED


## 5. Generate Train/Validation/Test Splits

In [11]:
# Only proceed with split generation if all tests passed
if all_tests_passed:
    print("All tests passed! Proceeding with split generation...")
    
    # Shuffle the files
    random.shuffle(obj_files)
    
    # Define split ratios
    train_ratio = 0.80
    val_ratio = 0.10
    test_ratio = 0.10
    
    # Calculate split indices
    train_split_index = int(len(obj_files) * train_ratio)
    val_split_index = train_split_index + int(len(obj_files) * val_ratio)
    
    # Create splits
    train_files = obj_files[:train_split_index]
    val_files = obj_files[train_split_index:val_split_index]
    test_files = obj_files[val_split_index:]
    
    # Create output directory if it doesn't exist
    output_dir = '../examples/splits/splits_hippocampus_only_healthy'
    os.makedirs(output_dir, exist_ok=True)
    
    # Save splits to JSON files
    with open(f'{output_dir}/train_split_hippocampus.json', 'w') as train_file:
        json.dump(train_files, train_file, indent=2)
    with open(f'{output_dir}/val_split_hippocampus.json', 'w') as val_file:
        json.dump(val_files, val_file, indent=2)
    with open(f'{output_dir}/test_split_hippocampus.json', 'w') as test_file:
        json.dump(test_files, test_file, indent=2)
    
    print(f"\n✓ Splits created from {len(obj_files)} files (first 510 entries from labels.pt):")
    print(f"  Train: {len(train_files)} files ({len(train_files)/len(obj_files)*100:.1f}%)")
    print(f"  Val: {len(val_files)} files ({len(val_files)/len(obj_files)*100:.1f}%)")
    print(f"  Test: {len(test_files)} files ({len(test_files)/len(obj_files)*100:.1f}%)")
    print(f"\n✓ Files saved to: {output_dir}/")
    
else:
    print("❌ Tests failed! Split generation skipped.")
    print("Please fix the data issues before generating splits.")

All tests passed! Proceeding with split generation...

✓ Splits created from 510 files (first 510 entries from labels.pt):
  Train: 408 files (80.0%)
  Val: 51 files (10.0%)
  Test: 51 files (10.0%)

✓ Files saved to: ../examples/splits/splits_hippocampus_only_healthy/
